In [219]:
import numpy as np
import pandas as pd


In [220]:
short = 50
med = 100
long = 200

In [221]:

df = pd.read_csv(r"historical prices\RSP_SPY.csv")

In [222]:
df.rename(columns={"RSP": "RSP eop"}, inplace=True)
df.rename(columns={"SPY": "SPY eop"}, inplace=True)

df['RSP bop'] = df["RSP eop"].shift(1)
df['SPY bop'] = df["SPY eop"].shift(1)

df['bop date'] = df['date'].shift(1)
df['eop date'] = df['date']

df['RSP rtn'] = np.log(df['RSP eop'] / df['RSP bop'])
df['SPY rtn'] = np.log(df['SPY eop'] / df['SPY bop'])

column_order = ['RSP bop', 'RSP eop', 'RSP rtn',
                'SPY bop', 'SPY eop', 'SPY rtn',
                'bop date', 'eop date']

df = df[column_order]

In [223]:
start = pd.Timestamp("2017-01-01")
end = pd.Timestamp("2026-08-15")

df["bop date"] = pd.to_datetime(df["bop date"], errors="coerce")
df["eop date"] = pd.to_datetime(df["eop date"], errors="coerce")

df = df.loc[df["bop date"].between(start, end)].copy()
df = df.loc[df["eop date"].between(start, end)].copy()


In [224]:
df['eop ratio'] = df['RSP eop'] / df['SPY eop']
df[f'eop {short}d ema'] = df['eop ratio'].ewm(span=short, adjust=False, min_periods=short).mean()
df[f'eop {med}d ema'] = df['eop ratio'].ewm(span=med, adjust=False, min_periods=med).mean()
df[f'eop {long}d ema'] = df['eop ratio'].ewm(span=long, adjust=False, min_periods=long).mean()


In [225]:
import numpy as np

uptrend = (
    (df[f"eop {short}d ema"] > df[f"eop {med}d ema"]) &
    (df[f"eop {med}d ema"] > df[f"eop {long}d ema"])
)

downtrend = (
    (df[f"eop {short}d ema"] < df[f"eop {med}d ema"]) &
    (df[f"eop {med}d ema"] < df[f"eop {long}d ema"])
)

df["trend"] = np.select(
    [uptrend, downtrend],
    [1, -1],
    default=0
)

df['position'] = df['trend'].shift(1)

df['daily strat rtn'] = df['position'] * (df['RSP rtn'] - df['SPY rtn'])
df['cum strat rtn'] = df['daily strat rtn'].cumsum()

In [226]:
df.to_csv(r"backtests\RSP_SPY.csv")